In [1]:
import numpy as np
import plotly.graph_objects as go

from photomesh.camera.colmap import load_colmap_dataset, parse_colmap_images
from photomesh.mesh_io import load_mesh

DATASET_DIR = "dataset/"
MESH_NAME = "poisson.ply"

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
import os

# ── Load mesh ────────────────────────────────────────────────
vertices, triangles = load_mesh(os.path.join(DATASET_DIR, MESH_NAME))
print(f"Mesh: {len(vertices)} verts, {len(triangles)} tris")

# ── Load COLMAP cameras ─────────────────────────────────────
cam_R, cam_t, cam_intr, image_paths = load_colmap_dataset(DATASET_DIR)
n_views = len(image_paths)

# Camera centers in world space: C = -R^T @ t
cam_centers = np.array([-R.T @ t for R, t in zip(cam_R, cam_t)])

# Image names for hover labels
image_entries = parse_colmap_images(os.path.join(DATASET_DIR, "images.txt"))
cam_labels = [os.path.basename(entry[4]) for entry in image_entries]

print(f"Cameras: {n_views}")
print(f"Camera centers range: {cam_centers.min(axis=0).round(3)} → {cam_centers.max(axis=0).round(3)}")

Mesh: 4766 verts, 9528 tris
Cameras: 9
Camera centers range: [-0.197 -0.28   0.   ] → [0.277 0.    0.324]


In [3]:
def build_frustum_lines(R_w2c, t_w2c, fx, fy, cx, cy, W, H, scale=0.05):
    """Build 3D line segments for a camera frustum.

    Returns arrays of x, y, z coordinates with None separators for
    Plotly Scatter3d line drawing.
    """
    # Camera center in world coords
    C = -R_w2c.T @ t_w2c

    # Four image-plane corners in camera coords (at z=1), then scaled
    corners_cam = np.array([
        [(0 - cx) / fx,  (0 - cy) / fy,  1.0],
        [(W - cx) / fx,  (0 - cy) / fy,  1.0],
        [(W - cx) / fx,  (H - cy) / fy,  1.0],
        [(0 - cx) / fx,  (H - cy) / fy,  1.0],
    ]) * scale

    # Transform to world coords
    corners_world = (R_w2c.T @ corners_cam.T).T + C

    # Line segments: center→each corner + rectangle around the image plane
    xs, ys, zs = [], [], []

    # Edges from center to corners
    for corner in corners_world:
        xs += [C[0], corner[0], None]
        ys += [C[1], corner[1], None]
        zs += [C[2], corner[2], None]

    # Rectangle around image plane
    for i in range(4):
        j = (i + 1) % 4
        xs += [corners_world[i, 0], corners_world[j, 0], None]
        ys += [corners_world[i, 1], corners_world[j, 1], None]
        zs += [corners_world[i, 2], corners_world[j, 2], None]

    return xs, ys, zs, corners_world

In [4]:
# ── Build the figure ─────────────────────────────────────────
fig = go.Figure()

# 1) Mesh (subsample triangles for performance)
max_tris = 50_000
if len(triangles) > max_tris:
    idx = np.random.default_rng(0).choice(len(triangles), max_tris, replace=False)
    tri_sub = triangles[idx]
else:
    tri_sub = triangles

fig.add_trace(go.Mesh3d(
    x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
    i=tri_sub[:, 0], j=tri_sub[:, 1], k=tri_sub[:, 2],
    color="lightblue", opacity=0.35,
    lighting=dict(ambient=0.6, diffuse=0.5),
    name="Mesh",
    hoverinfo="skip",
))

# 2) Camera centers
fig.add_trace(go.Scatter3d(
    x=cam_centers[:, 0], y=cam_centers[:, 1], z=cam_centers[:, 2],
    mode="markers+text",
    marker=dict(size=5, color="red"),
    text=cam_labels,
    textposition="top center",
    textfont=dict(size=9),
    name="Cameras",
    hovertext=cam_labels,
    hoverinfo="text",
))

# 3) Camera frustums
colors = [
    "#e6194b", "#3cb44b", "#4363d8", "#f58231", "#911eb4",
    "#42d4f4", "#f032e6", "#bfef45", "#fabed4",
]

for i in range(n_views):
    fx, fy, cx, cy, W, H = cam_intr[i]
    xs, ys, zs, _ = build_frustum_lines(cam_R[i], cam_t[i], fx, fy, cx, cy, W, H)
    color = colors[i % len(colors)]
    fig.add_trace(go.Scatter3d(
        x=xs, y=ys, z=zs,
        mode="lines",
        line=dict(color=color, width=3),
        name=cam_labels[i],
        hoverinfo="name",
        showlegend=False,
    ))

# ── Layout ───────────────────────────────────────────────────
fig.update_layout(
    title="Mesh + COLMAP Cameras",
    scene=dict(
        aspectmode="data",
        xaxis_title="X", yaxis_title="Y", zaxis_title="Z",
    ),
    width=1000, height=750,
    legend=dict(x=0.01, y=0.99),
)

fig.show()